In [0]:
%run ../00-common/01.enviroment-config

In [0]:

bronze_table=f"{catalog_name}.{bronze_schema}.circuits"
silver_table=f"{catalog_name}.{silver_schema}.circuits"

In [0]:

#circuits_df=spark.read.option('versionAsOf',0).table(bronze_table)
    


In [0]:

circuits_df=spark.table(bronze_table)
    


In [0]:
display(circuits_df)

In [0]:
circuits_selected_df=circuits_df.select('circuitId','circuitName','lat','long','locality','country','ingestion_timestamp','source_file')

In [0]:
from pyspark.sql import functions as F

In [0]:
circuits_selected_df=circuits_df.select(
                                        F.col('circuitId'), 
                                        F.col('circuitName'),  
                                        F.col('lat'), 
                                        F.col('long'), 
                                        F.col('locality'),  
                                        F.col('country'), 
                                        F.col('ingestion_timestamp'), 
                                        F.col('source_file')
                                        )


In [0]:
# circuits_renames_df=( circuits_selected_df
#                         .withColumnRenamed('circuitId','circuit_id')
#                         .withColumnRenamed('circuitName','circuit_name')
#                         .withColumnRenamed('lat','latitude')
#                         .withColumnRenamed('long','longitude')                     
                       
#                       )



In [0]:
circuits_renamed_df=(
    circuits_selected_df
    .withColumnsRenamed({"circuitId":"circuit_id","circuitName":"circuit_name","lat":"latitude","long":"longitude"})
)

In [0]:
# circuits_valid_df=circuits_renamed_df.filter("circuit_id IS NOT NULL")

In [0]:
circuits_valid_df=circuits_renamed_df.filter(F.col('circuit_id').isNotNull())


In [0]:
# circuits_distinct_df=circuits_valid_df.distinct()


In [0]:
circuits_distinct_df=circuits_valid_df.dropDuplicates(['circuit_id'])

In [0]:
circuits_final_df=(circuits_distinct_df
    .withColumn('circuit_name',F.initcap(F.col("circuit_name")))
    .withColumn('locality',F.initcap(F.col("locality"))))

In [0]:
display(circuits_final_df)

In [0]:
(circuits_final_df.write
                .mode('overwrite')
                .format('delta')
                .saveAsTable(silver_table)
                )

In [0]:
display(spark.table(silver_table))